# GPT-OSS-20B quick notebook

This notebook shows a minimal example of calling the `gpt-oss-20b` model using the OpenAI Python client.

IMPORTANT: Do not put your API key in the notebook. Set the environment variable `OPENAI_API_KEY` for this project (e.g. in your shell, or in a `.env` file loaded with python-dotenv).

In [23]:
import sys
import subprocess

# Install runtime deps into the current environment if needed
subprocess.check_call([sys.executable, "-m", "pip", "install", "openai", "python-dotenv" ,"--quiet"])

0

In [24]:
# Load environment variables from a .env file if present
from dotenv import load_dotenv
from pathlib import Path
import os

# Try project root .env (not required)
root = Path.cwd().parents[0]
env_path = Path.cwd() / '.env'
if not env_path.exists() and (root / '.env').exists():
    env_path = root / '.env'
load_dotenv(dotenv_path=env_path)

print(f"Loaded .env from: {env_path if env_path.exists() else 'none'}")

Loaded .env from: c:\Users\mathe\Documents\GitHub\llm_engineering\.env


In [25]:
from openai import OpenAI
import os

# Use Hugging Face Inference Router via the OpenAI-compatible client.
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not set. Set it in your environment or .env file (or export HF_TOKEN).")

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)
print("OpenAI client created via Hugging Face Inference Router (HF_TOKEN loaded).")

OpenAI client created via Hugging Face Inference Router (HF_TOKEN loaded).


In [29]:
# Chat completions version: put the system prompt as a "system" role message
def call_gpt_oss_20b_chat(prompt: str, system_prompt: str | None = None, max_output_tokens: int = 512, temperature: float = 0.2):
    try:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        resp = client.chat.completions.create(
            model="openai/gpt-oss-20b:fireworks-ai",
            messages=messages,
            temperature=temperature,
            max_tokens=max_output_tokens,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"ERROR: {e!s}"

# Responses API version: pass 'instructions' param
def call_gpt_oss_20b_responses(prompt: str, instructions: str | None = None, max_output_tokens: int = 512, temperature: float = 0.2):
    try:
        resp = client.responses.create(
            model="openai/gpt-oss-20b",   # or "openai/gpt-oss-20b:fireworks-ai" depending on router mapping
            input=prompt,
            instructions=instructions,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )
        # resp.output_text concatenates text outputs; adjust if you need message objects
        return resp.output_text
    except Exception as e:
        return f"ERROR: {e!s}"

# Example usage

system_prompt = "You are a helpful assistant that only speaks Portuguese from Brazil."
user_prompt = "Summarize the benefits of using small, focused unit tests in Python."
print(call_gpt_oss_20b_responses(user_prompt, instructions=system_prompt))

**Benefícios de usar testes unitários pequenos e focados em Python**

- **Diagnóstico rápido**  
  Quando um teste falha, ele aponta exatamente a função ou linha que está quebrada, facilitando a correção.

- **Manutenção mais simples**  
  Testes pequenos dependem de poucos parâmetros e de um único comportamento, tornando‑se mais fáceis de atualizar quando a lógica muda.

- **Cobertura mais efetiva**  
  Pequenos testes cobrem cenários específicos (ex.: entrada válida, entrada inválida, exceções), garantindo que cada caminho de execução seja verificado.

- **Isolamento de dependências**  
  Testes focados permitem usar *mocks* ou *fixtures* de forma mais clara, isolando a unidade de código das dependências externas.

- **Velocidade de execução**  
  Testes menores consomem menos recursos e são executados mais rapidamente, possibilitando rodar o conjunto completo de testes em cada commit.

- **Facilidade de leitura e documentação**  
  Um teste que descreve apenas um comportamento serve

Notes:
- If the model name `gpt-oss-20b` is unavailable or different in your OpenAI deployment, change the `model` parameter accordingly.
- Keep your API key secret. Use environment variables or a secure secrets manager.
- For production, add retries, backoff, and robust error handling.